In [1]:
"""
4. R-CNN (Region-based Convolutional Neural Network)
=====================================================
R-CNN was proposed by Girshick et al. (2014).

Pipeline:
  1. Generate ~2000 region proposals using Selective Search
  2. Resize each region to fixed size and extract CNN features
  3. Classify each region using SVMs
  4. Refine bounding boxes using regression

This demo uses Selective Search + pre-trained ResNet features.
"""

import cv2
import numpy as np
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torchvision.models as models
    import torchvision.transforms as transforms
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not installed. Install: pip install torch torchvision")


def simple_region_proposals(image, num_proposals=100):
    """Generate region proposals using a grid-based approach."""
    h, w = image.shape[:2]
    proposals = []
    for scale in [0.2, 0.3, 0.5]:
        bw, bh = int(w * scale), int(h * scale)
        for y in range(0, h - bh, bh // 3):
            for x in range(0, w - bw, bw // 3):
                proposals.append([x, y, bw, bh])
    if len(proposals) > num_proposals:
        idx = np.random.choice(len(proposals), num_proposals, replace=False)
        proposals = [proposals[i] for i in idx]
    return np.array(proposals)


class CNNFeatureExtractor:
    """Extract features using pre-trained ResNet-18."""
    def __init__(self):
        if not TORCH_AVAILABLE:
            self.model = None
            return
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.model = nn.Sequential(*list(resnet.children())[:-1])
        self.model.eval()
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def extract(self, region):
        if self.model is None:
            return np.random.randn(512)
        tensor = self.transform(region).unsqueeze(0)
        with torch.no_grad():
            return self.model(tensor).squeeze().numpy()


def nms(boxes, scores, threshold=0.3):
    """Non-Maximum Suppression."""
    if len(boxes) == 0:
        return [], []
    boxes = np.array(boxes, dtype=np.float32)
    scores = np.array(scores)
    x1, y1 = boxes[:, 0], boxes[:, 1]
    x2, y2 = x1 + boxes[:, 2], y1 + boxes[:, 3]
    areas = boxes[:, 2] * boxes[:, 3]
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)
        overlap = (w * h) / (areas[order[1:]] + 1e-6)
        inds = np.where(overlap < threshold)[0]
        order = order[inds + 1]
    return boxes[keep].astype(int), scores[keep]


def rcnn_detect(image, max_proposals=50, score_threshold=0.4):
    """Complete R-CNN detection pipeline."""
    print("   Step 1: Generating region proposals...")
    try:
        ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
        ss.setBaseImage(image)
        ss.switchToSelectiveSearchFast()
        proposals = ss.process()[:max_proposals]
        print(f"   {len(proposals)} proposals (Selective Search)")
    except (cv2.error, AttributeError):
        proposals = simple_region_proposals(image, max_proposals)
        print(f"   {len(proposals)} proposals (Grid fallback)")

    print("   Step 2: Extracting CNN features...")
    extractor = CNNFeatureExtractor()
    features, valid = [], []
    for (x, y, w, h) in proposals:
        x, y, w, h = int(x), int(y), int(w), int(h)
        if w < 10 or h < 10:
            continue
        region = image[y:y+h, x:x+w]
        if region.size == 0:
            continue
        features.append(extractor.extract(region))
        valid.append([x, y, w, h])

    if not features:
        return np.array([]), np.array([])

    features = np.array(features)
    print(f"   Features extracted: {len(features)}, dim={features.shape[1]}")

    print("   Step 3: Scoring regions...")
    scores = np.array([1.0/(1.0+np.exp(-np.mean(np.abs(f))/np.std(f)+3))
                        for f in features])

    mask = scores > score_threshold
    valid = np.array(valid)
    print("   Step 4: NMS...")
    final_boxes, final_scores = nms(valid[mask], scores[mask])
    print(f"   Final detections: {len(final_boxes)}")
    return final_boxes, final_scores


if __name__ == "__main__":
    print("=" * 60)
    print("R-CNN OBJECT DETECTION")
    print("=" * 60)

    # Create test image
    img = np.ones((400, 600, 3), dtype=np.uint8) * 220
    cv2.rectangle(img, (50, 50), (200, 180), (200, 100, 50), -1)
    cv2.circle(img, (400, 150), 70, (50, 180, 50), -1)
    pts = np.array([[300, 350], [350, 250], [400, 350]], np.int32)
    cv2.fillPoly(img, [pts], (50, 50, 200))
    print(f"\nTest image: {img.shape}")

    boxes, scores = rcnn_detect(img)
    output = img.copy()
    for box, sc in zip(boxes, scores):
        x, y, w, h = box
        cv2.rectangle(output, (x,y), (x+w,y+h), (0,255,0), 2)
        cv2.putText(output, f"{sc:.2f}", (x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
    cv2.imwrite("rcnn_result.jpg", output)
    print(f"\nSaved: rcnn_result.jpg")

    print("\n--- How R-CNN Works ---")
    print("1. Selective Search generates ~2000 region proposals")
    print("2. Each region → CNN → 4096-dim feature vector")
    print("3. Class-specific SVMs classify each region")
    print("4. Bounding box regression refines coordinates")
    print("\nLimitations: ~47s/image, multi-stage, disk-intensive")
    print("Done!")

R-CNN OBJECT DETECTION

Test image: (400, 600, 3)
   Step 1: Generating region proposals...
   50 proposals (Selective Search)
   Step 2: Extracting CNN features...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 227MB/s]


   Features extracted: 23, dim=512
   Step 3: Scoring regions...
   Step 4: NMS...
   Final detections: 0

Saved: rcnn_result.jpg

--- How R-CNN Works ---
1. Selective Search generates ~2000 region proposals
2. Each region → CNN → 4096-dim feature vector
3. Class-specific SVMs classify each region
4. Bounding box regression refines coordinates

Limitations: ~47s/image, multi-stage, disk-intensive
Done!
